# Phenotype ↔ ChemicalEntity Relation-Wise Merge

Merges Phenotype–Chemical triples from PrimeKG; fills missing `tail_detail_name`
via PubChem IUPAC lookup; deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [38]:
import os
import pandas as pd

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PHENOTYPE_CHEMICALENTITY/ALL_PHENOTYPE_CHEMICALENTITY.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Chemical Lookup Dictionary — PubChem

In [2]:
Pubchem = pd.read_pickle(DB_DIR + 'pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem
print(f"PubChem IUPAC dict: {len(Pubchem_IUPAC_CID_Dict):,} entries")

PubChem IUPAC dict: 119,177,440 entries


## 2. Load KG Sources

### 2a. PrimeKG

In [30]:
PrimeKG_Phenotype_Chemical = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Phenotype_Chemical_1.csv')
PrimeKG_Phenotype_Chemical.columns = PrimeKG_Phenotype_Chemical.columns.str.lower()

# Fill missing tail_detail_name via PubChem lookup
mask = PrimeKG_Phenotype_Chemical['tail_detail_name'].isna()
if mask.any():
    PrimeKG_Phenotype_Chemical.loc[mask, 'tail_detail_name'] = (
        PrimeKG_Phenotype_Chemical.loc[mask, 'tail'].map(Pubchem_IUPAC_CID_Dict)
    )
print(f"PrimeKG: {len(PrimeKG_Phenotype_Chemical):,} rows | missing tail_detail_name: {PrimeKG_Phenotype_Chemical['tail_detail_name'].isna().sum():,}")
PrimeKG_Phenotype_Chemical['kg_type'] = 'Generalised'
PrimeKG_Phenotype_Chemical['species'] = 'Homo species'

PrimeKG_Phenotype_Chemical.head(2)

PrimeKG: 64,784 rows | missing tail_detail_name: 0


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,tail_pubchem_id,tail_smiles_name,tail_smiles_name.1,kg_type,species
0,HP:0002027,Phenotype_ChemicalEntity,10917,Phenotype,side effect,ChemicalEntity,PrimeKG,HPO_ID,Pubchem,Abdominal pain,(3R)-3-hydroxy-4-(trimethylazaniumyl)butanoate,Levocarnitine,C[N+](C)(C)C[C@@H](CC(=O)[O-])O,C[N+](C)(C)C[C@@H](CC(=O)[O-])O,Generalised,Homo species
1,HP:0004396,Phenotype_ChemicalEntity,10917,Phenotype,side effect,ChemicalEntity,PrimeKG,HPO_ID,Pubchem,Poor appetite,(3R)-3-hydroxy-4-(trimethylazaniumyl)butanoate,Levocarnitine,C[N+](C)(C)C[C@@H](CC(=O)[O-])O,C[N+](C)(C)C[C@@H](CC(=O)[O-])O,Generalised,Homo species


### 2a. Monarch kg

In [31]:
monarch = pd.read_csv(PROC_DIR + 'Monarch/Human/Human_Phenotype_ChemicalEntity_Monarch.csv')
monarch.columns = monarch.columns.str.lower()
monarch.rename(columns={'head_name': 'head_detail_name', 'tail_name': 'tail_detail_name'}, inplace=True)
monarch['kg_type'] = 'Generalised'
monarch['kg_source'] = 'MonarchKG'
monarch['species'] = 'Homo species'
monarch

,head,tail,relation_type,relation_source,kgsource,head_detail_name,head_type,head_id_is,head_taxon,head_taxon_label,...,head_taxon_name,tail_taxon_name,head_type_clean,tail_type_clean,relation,species_species,tail_id,kg_type,kg_source,species
0,UPHENO:0066726,586,related_to,infores:upheno,MonarchKG,decreased level of creatine,PhenotypicFeature,UPHENO,NaN,NaN,...,Homo sapiens,NaN,PhenotypicFeature,ChemicalEntity,PhenotypicFeature_ChemicalEntity,NaN,CHEBI:16919,Generalised,MonarchKG,Homo species
1,UPHENO:0066727,5961,related_to,infores:upheno,MonarchKG,decreased level of glutamine,PhenotypicFeature,UPHENO,NaN,NaN,...,Homo sapiens,NaN,PhenotypicFeature,ChemicalEntity,PhenotypicFeature_ChemicalEntity,NaN,CHEBI:28300,Generalised,MonarchKG,Homo species
2,UPHENO:0066729,750,related_to,infores:upheno,MonarchKG,decreased level of glycine,PhenotypicFeature,UPHENO,NaN,NaN,...,Homo sapiens,NaN,PhenotypicFeature,ChemicalEntity,PhenotypicFeature_ChemicalEntity,NaN,CHEBI:15428,Generalised,MonarchKG,Homo species
3,UPHENO:0066730,5280335,related_to,infores:upheno,MonarchKG,decreased level of sphingosine,PhenotypicFeature,UPHENO,NaN,NaN,...,Homo sapiens,NaN,PhenotypicFeature,ChemicalEntity,PhenotypicFeature_ChemicalEntity,NaN,CHEBI:16393,Generalised,MonarchKG,Homo species
4,UPHENO:0066733,853,related_to,infores:upheno,MonarchKG,decreased level of thyroxine,PhenotypicFeature,UPHENO,NaN,NaN,...,Homo sapiens,NaN,PhenotypicFeature,ChemicalEntity,PhenotypicFeature_ChemicalEntity,NaN,CHEBI:30660,Generalised,MonarchKG,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1110,UPHENO:0052029,232,related_to,infores:upheno,MonarchKG,cerebrospinal fluid arginine level phenotype,PhenotypicFeature,UPHENO,NaN,NaN,...,Homo sapiens,NaN,PhenotypicFeature,ChemicalEntity,PhenotypicFeature_ChemicalEntity,NaN,CHEBI:29016,Generalised,MonarchKG,Homo species
1111,UPHENO:0052032,5192,related_to,infores:upheno,MonarchKG,urine sebacic acid level phenotype,PhenotypicFeature,UPHENO,NaN,NaN,...,Homo sapiens,NaN,PhenotypicFeature,ChemicalEntity,PhenotypicFeature_ChemicalEntity,NaN,CHEBI:41865,Generalised,MonarchKG,Homo species
1112,UPHENO:0052033,681,related_to,infores:upheno,MonarchKG,cerebrospinal fluid dopamine level phenotype,PhenotypicFeature,UPHENO,NaN,NaN,...,Homo sapiens,NaN,PhenotypicFeature,ChemicalEntity,PhenotypicFeature_ChemicalEntity,NaN,CHEBI:18243,Generalised,MonarchKG,Homo species
1113,UPHENO:0052036,750,related_to,infores:upheno,MonarchKG,cerebrospinal fluid glycine level phenotype,PhenotypicFeature,UPHENO,NaN,NaN,...,Homo sapiens,NaN,PhenotypicFeature,ChemicalEntity,PhenotypicFeature_ChemicalEntity,NaN,CHEBI:15428,Generalised,MonarchKG,Homo species


In [32]:
monarch.columns

Index(['head', 'tail', 'relation_type', 'relation_source', 'kgsource',
       'head_detail_name', 'head_type', 'head_id_is', 'head_taxon',
       'head_taxon_label', 'tail_detail_name', 'tail_type', 'tail_id_is',
       'tail_taxon', 'tail_taxon_label', 'head_taxon_name', 'tail_taxon_name',
       'head_type_clean', 'tail_type_clean', 'relation', 'species_species',
       'tail_id', 'kg_type', 'kg_source', 'species'],
      dtype='object')

## 3. Check and Fix Duplicate Columns

In [33]:
SOURCE_DFS = [('PrimeKG_Phenotype_Chemical', PrimeKG_Phenotype_Chemical),
                ('monarch', monarch)]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[PrimeKG_Phenotype_Chemical] ✓ no duplicates
[monarch] ✓ no duplicates


## 4. Align Schemas and Concatenate

In [34]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 65,899 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,HP:0002027,Phenotype_ChemicalEntity,10917,Phenotype,side effect,ChemicalEntity,PrimeKG,HPO_ID,Pubchem,Abdominal pain,(3R)-3-hydroxy-4-(trimethylazaniumyl)butanoate,Generalised,Homo species
1,HP:0004396,Phenotype_ChemicalEntity,10917,Phenotype,side effect,ChemicalEntity,PrimeKG,HPO_ID,Pubchem,Poor appetite,(3R)-3-hydroxy-4-(trimethylazaniumyl)butanoate,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [35]:
final_df['head_type']  = 'Phenotype'
final_df['tail_type']  = 'ChemicalEntity'
final_df['relation']   = 'Phenotype_ChemicalEntity'
final_df['head_id_is'] = 'HPO'
final_df['tail_id_is'] = 'Pubchem'

print("Unique relation:",  set(final_df['relation']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Phenotype_ChemicalEntity'}
Unique kg_source: {'PrimeKG', 'MonarchKG'}


## 6. Deduplicate and Add Schema Columns

In [36]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

final_df_dedup = final_df_dedup[REQUIRED_COLS]
final_df_dedup.head()

After deduplication: 65,899 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,HP:0000016,Phenotype_ChemicalEntity,10297,Phenotype,side effect,ChemicalEntity,PrimeKG,HPO,Pubchem,Urinary retention,"(1R,2S)-2-amino-1-phenylpropan-1-ol",Generalised,Homo species
1,HP:0000016,Phenotype_ChemicalEntity,107807,Phenotype,side effect,ChemicalEntity,PrimeKG,HPO,Pubchem,Urinary retention,"(2S,3aS,7aS)-1-[(2S)-2-[[(2S)-1-ethoxy-1-oxope...",Generalised,Homo species
2,HP:0000016,Phenotype_ChemicalEntity,115237,Phenotype,side effect,ChemicalEntity,PrimeKG,HPO,Pubchem,Urinary retention,"3-[2-[4-(6-fluoro-1,2-benzoxazol-3-yl)piperidi...",Generalised,Homo species
3,HP:0000016,Phenotype_ChemicalEntity,11622909,Phenotype,side effect,ChemicalEntity,PrimeKG,HPO,Pubchem,Urinary retention,"2-(aminomethyl)-N,N-diethyl-1-phenylcyclopropa...",Generalised,Homo species
4,HP:0000016,Phenotype_ChemicalEntity,119570,Phenotype,side effect,ChemicalEntity,PrimeKG,HPO,Pubchem,Urinary retention,"(6S)-6-N-propyl-4,5,6,7-tetrahydro-1,3-benzoth...",Generalised,Homo species


## 7. QC — NaN Check

In [37]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })
    

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 8. Save Output

In [39]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 65,899 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PHENOTYPE_CHEMICALENTITY/ALL_PHENOTYPE_CHEMICALENTITY.csv
